# Day 10 — Linear regression, loss functions and baseline models

Run each cell with **Shift + Enter**, from the top. We use the 150-student dataset from Days 5-9, `student_data_150.csv`: 8 real students and 142 **synthetic** ones (simulated for practice, marked in the Source column).

## 1. Understanding linear regression

In [ ]:
# Get the dataset: Colab starts with an empty folder, so download the file the first time
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-010/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# The same 150 students as Days 5-9: 8 real ones and 142 synthetic ones
df = pd.read_csv("student_data_150.csv")

# Check for missing values before fitting: LinearRegression can't handle them
print("Missing values in the columns we use:", df[["StudyHours", "ExamScore"]].isna().sum().sum())

# Hold back a validation set (to compare models) and a test set (for the final check only)
train_val, test = train_test_split(df, test_size=0.15, random_state=42)
train, validation = train_test_split(train_val, test_size=0.15 / 0.85, random_state=42)
print("Train", len(train), "| validation", len(validation), "| test", len(test), "(kept aside)")

Missing values in the columns we use: 0
Train 104 | validation 23 | test 23 (kept aside)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# One feature (StudyHours) and the target (ExamScore)
model = LinearRegression()
model.fit(train[["StudyHours"]], train["ExamScore"])
predictions = model.predict(validation[["StudyHours"]])

print(f"Slope (coef_): {model.coef_[0]:.2f}")
print(f"Intercept (intercept_): {model.intercept_:.2f}")
print(f"Validation MAE: {mean_absolute_error(validation['ExamScore'], predictions):.2f}")
print(f"Validation R2: {r2_score(validation['ExamScore'], predictions):.3f}")

Slope (coef_): 6.18
Intercept (intercept_): 34.86
Validation MAE: 9.78
Validation R2: 0.236


In [ ]:
# Use the line to predict: score = slope * hours + intercept
new_student = pd.DataFrame({"StudyHours": [6]})
print(f"Predicted score for 6 study hours: {model.predict(new_student)[0]:.1f}")
print(f"By hand: {model.coef_[0]:.2f} * 6 + {model.intercept_:.2f} = {model.coef_[0] * 6 + model.intercept_:.1f}")

# The line goes on forever, but the data doesn't: 0 hours and 20 hours are outside what it saw
print("Study hours in the training data:", train["StudyHours"].min(), "to", train["StudyHours"].max())
train[["StudyHours", "ExamScore"]].to_csv("training_data.csv", index=False)

Predicted score for 6 study hours: 71.9
By hand: 6.18 * 6 + 34.86 = 71.9
Study hours in the training data: 0.5 to 7.0


In [ ]:
# More than one feature: one coefficient per feature, plus the intercept
features = ["StudyHours", "Attendance", "Age"]
multi = LinearRegression().fit(train[features], train["ExamScore"])
print(f"b0 (intercept): {multi.intercept_:.2f}")
for name, b in zip(features, multi.coef_):
    print(f"{name:<11} {b:+.2f}")
print(f"Validation MAE: {mean_absolute_error(validation['ExamScore'], multi.predict(validation[features])):.2f}")

b0 (intercept): -0.70
StudyHours  +5.49
Attendance  +0.39
Age         +0.64
Validation MAE: 8.83


## 2. The model: coefficients, intercept and predictions

In [ ]:
# Section 2: a model with two features, fitted on the training students only
features = ["StudyHours", "Attendance"]
model2 = LinearRegression().fit(train[features], train["ExamScore"])
b0, (b1, b2) = model2.intercept_, model2.coef_
print(f"Intercept (b0): {b0:.2f}")
print(f"Coefficients: StudyHours (b1) {b1:.2f} | Attendance (b2) {b2:.2f}")
print(f"Equation: ExamScore = {b0:.2f} + {b1:.2f} x StudyHours + {b2:.2f} x Attendance")

Intercept (b0): 6.61
Coefficients: StudyHours (b1) 5.55 | Attendance (b2) 0.38
Equation: ExamScore = 6.61 + 5.55 x StudyHours + 0.38 x Attendance


In [ ]:
# Predict for new students: the equation by hand, then predict()
new_students = pd.DataFrame({"StudyHours": [6, 4], "Attendance": [85, 90]})
for (_, s), p in zip(new_students.iterrows(), model2.predict(new_students)):
    by_hand = b0 + b1 * s["StudyHours"] + b2 * s["Attendance"]
    print(f"{s['StudyHours']:.0f} hours, {s['Attendance']:.0f}% attendance: "
          f"{b0:.2f} + {b1:.2f} x {s['StudyHours']:.0f} + {b2:.2f} x {s['Attendance']:.0f} = {by_hand:.1f} | predict(): {p:.1f}")

6 hours, 85% attendance: 6.61 + 5.55 x 6 + 0.38 x 85 = 72.3 | predict(): 72.3
4 hours, 90% attendance: 6.61 + 5.55 x 4 + 0.38 x 90 = 63.2 | predict(): 63.2


In [ ]:
# Coefficients depend on the other features in the model...
one = LinearRegression().fit(train[["StudyHours"]], train["ExamScore"])
print(f"StudyHours coefficient alone: {one.coef_[0]:.2f} | with Attendance: {b1:.2f}")
# ...and on the units: attendance as a fraction (0.85) instead of a percentage (85)
as_fraction = train[features].assign(Attendance=train["Attendance"] / 100)
scaled = LinearRegression().fit(as_fraction, train["ExamScore"])
print(f"Attendance coefficient per percentage point: {b2:.2f} | per whole 0-1 unit: {scaled.coef_[1]:.2f}")
print(f"Same predictions either way: {abs(scaled.predict(as_fraction) - model2.predict(train[features])).max() < 1e-9}")

StudyHours coefficient alone: 6.18 | with Attendance: 5.55
Attendance coefficient per percentage point: 0.38 | per whole 0-1 unit: 38.17
Same predictions either way: True


In [ ]:
# Does the second feature help? Compare on the validation students
for name, m, cols in [("StudyHours only", one, ["StudyHours"]), ("StudyHours + Attendance", model2, features)]:
    p = m.predict(validation[cols])
    print(f"{name:<24} validation MAE {mean_absolute_error(validation['ExamScore'], p):.2f} "
          f"| R2 {r2_score(validation['ExamScore'], p):.3f}")
pd.DataFrame({"ActualScore": validation["ExamScore"],
              "PredictedScore": model2.predict(validation[features]).round(1)}).to_csv("validation_predictions.csv", index=False)

StudyHours only          validation MAE 9.78 | R2 0.236
StudyHours + Attendance  validation MAE 8.88 | R2 0.309


## 3. Loss functions: MAE, MSE and RMSE

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Section 3: the worked example, three students (illustrative numbers)
y_true = [80, 60, 90]
y_pred = [75, 65, 85]

mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)               # newer scikit-learn also has root_mean_squared_error()

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)

MAE: 5.0
MSE: 25.0
RMSE: 5.0


In [ ]:
# The same metrics by hand: error, absolute error and squared error per student
errors = np.array(y_true) - np.array(y_pred)
print(pd.DataFrame({"Actual": y_true, "Predicted": y_pred, "Error": errors,
                    "AbsError": np.abs(errors), "SqError": errors ** 2}).to_string(index=False))
print(f"MAE = mean(|error|) = {np.abs(errors).mean():.2f} | MSE = mean(error^2) = {(errors ** 2).mean():.2f}"
      f" | RMSE = sqrt(MSE) = {np.sqrt((errors ** 2).mean()):.2f}")

 Actual  Predicted  Error  AbsError  SqError
     80         75      5         5       25
     60         65     -5         5       25
     90         85      5         5       25
MAE = mean(|error|) = 5.00 | MSE = mean(error^2) = 25.00 | RMSE = sqrt(MSE) = 5.00


In [ ]:
# Practice: one large error moves MSE much more than MAE
def report(name, actual, predicted):
    mse = mean_squared_error(actual, predicted)
    print(f"{name:<22} MAE {mean_absolute_error(actual, predicted):6.2f} | MSE {mse:6.2f} | RMSE {np.sqrt(mse):6.2f}")

actual = [50, 70, 90]
report("Errors 5, -5, 10", actual, [45, 75, 80])
report("Errors 5, -5, 20", actual, [45, 75, 70])     # the third prediction is now 20 marks off

Errors 5, -5, 10       MAE   6.67 | MSE  50.00 | RMSE   7.07
Errors 5, -5, 20       MAE  10.00 | MSE 150.00 | RMSE  12.25


In [ ]:
# Absolute vs squared error for errors from 0 to 10 marks
curve = pd.DataFrame({"Error": range(0, 11)})
curve["AbsoluteError"] = curve["Error"].abs()
curve["SquaredError"] = curve["Error"] ** 2
print(curve.T.to_string(header=False))
curve.to_csv("loss_curve.csv", index=False)

Error          0  1  2  3   4   5   6   7   8   9   10
AbsoluteError  0  1  2  3   4   5   6   7   8   9   10
SquaredError   0  1  4  9  16  25  36  49  64  81  100


In [ ]:
# The two-feature model from Section 2, on the validation students
p = model2.predict(validation[features])
mse = mean_squared_error(validation["ExamScore"], p)
print(f"MAE  {mean_absolute_error(validation['ExamScore'], p):.2f} marks")
print(f"MSE  {mse:.2f} squared marks")
print(f"RMSE {np.sqrt(mse):.2f} marks")

MAE  8.88 marks
MSE  110.76 squared marks
RMSE 10.52 marks


In [ ]:
from sklearn.linear_model import QuantileRegressor

# Training objective vs evaluation metric: LinearRegression minimizes squared error on the training data
def train_scores(name, m):
    p = m.predict(train[features])
    print(f"{name:<26} training MSE {mean_squared_error(train['ExamScore'], p):6.2f} | MAE {mean_absolute_error(train['ExamScore'], p):5.2f}")

train_scores("LinearRegression (OLS)", model2)
# A median regression minimizes absolute error instead: lower training MAE, higher training MSE
median = QuantileRegressor(quantile=0.5, alpha=0, solver="highs").fit(train[features], train["ExamScore"])
train_scores("Median regression (MAE)", median)

LinearRegression (OLS)     training MSE 111.56 | MAE  8.26
Median regression (MAE)    training MSE 113.03 | MAE  8.23


## 4. Baseline models

In [ ]:
# Section 4: baselines. A tiny example first: the mean of five scores
scores = pd.Series([60, 70, 80, 90, 100])
print("Mean of 60, 70, 80, 90, 100:", scores.mean(), "-> the mean baseline predicts this for every student")

# On our data: statistics from the TRAINING students only
print(f"Training ExamScore mean: {train['ExamScore'].mean():.2f} | median: {train['ExamScore'].median():.2f}")

Mean of 60, 70, 80, 90, 100: 80.0 -> the mean baseline predicts this for every student
Training ExamScore mean: 57.08 | median: 57.00


In [ ]:
from sklearn.dummy import DummyRegressor

# Three baselines and a trained model, all scored on the same validation students
features3 = ["StudyHours", "Attendance", "Age"]
X_train, y_train = train[features3], train["ExamScore"]
X_val, y_val = validation[features3], validation["ExamScore"]

predictions = {
    "Mean baseline": DummyRegressor(strategy="mean").fit(X_train, y_train).predict(X_val),
    "Median baseline": DummyRegressor(strategy="median").fit(X_train, y_train).predict(X_val),
    "Rule: 5 x hours + 40": 5 * X_val["StudyHours"] + 40,       # a simple domain rule, no learning
    "Linear regression": LinearRegression().fit(X_train, y_train).predict(X_val),
}
rows = []
for name, p in predictions.items():
    mae, rmse, r2 = mean_absolute_error(y_val, p), np.sqrt(mean_squared_error(y_val, p)), r2_score(y_val, p)
    rows.append({"Model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "R2": round(r2, 3)})
results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv("baseline_comparison.csv", index=False)

               Model   MAE  RMSE     R2
       Mean baseline 10.59 12.74 -0.012
     Median baseline 10.61 12.75 -0.014
Rule: 5 x hours + 40  9.39 10.98  0.249
   Linear regression  8.83 10.50  0.312


In [ ]:
# The mean baseline uses the TRAINING mean, so its R2 on other students isn't exactly 0
print(f"Validation mean: {y_val.mean():.2f} vs training mean: {y_train.mean():.2f}")
improvement = 1 - results.loc[3, "MAE"] / results.loc[0, "MAE"]
print(f"Linear regression lowers the mean baseline's validation MAE by {improvement:.0%}")
print("Same validation students for every model:", all(len(p) == len(y_val) for p in predictions.values()))

Validation mean: 58.48 vs training mean: 57.08
Linear regression lowers the mean baseline's validation MAE by 17%
Same validation students for every model: True


## 5. End-to-end example: predicting exam scores

In [ ]:
# Section 5, end to end. Step 1: explore the data
df = pd.read_csv("student_data_150.csv")
print("Students:", len(df), "| missing values:", int(df.isnull().sum().sum()))
print(f"Average exam score: {df['ExamScore'].mean():.1f} | average study hours: {df['StudyHours'].mean():.1f}")
print(df[["StudyHours", "Attendance", "Age", "ExamScore"]].describe().round(1).loc[["mean", "std", "min", "max"]])
print("\nCorrelation with ExamScore:", df[["StudyHours", "Attendance", "Age"]].corrwith(df["ExamScore"]).round(2).to_dict())

Students: 150 | missing values: 0
Average exam score: 57.4 | average study hours: 3.6
      StudyHours  Attendance   Age  ExamScore
mean         3.6        79.5  11.0       57.4
std          1.4        10.0   0.8       13.9
min          0.5        54.0  10.0       24.0
max          7.0       100.0  12.0       91.0

Correlation with ExamScore: {'StudyHours': 0.61, 'Attendance': 0.38, 'Age': 0.08}


In [ ]:
# Step 2: split once (70/15/15), then choose on the validation set
train_val, test = train_test_split(df, test_size=0.15, random_state=42)
train, validation = train_test_split(train_val, test_size=0.15 / 0.85, random_state=42)

candidates = {
    "Mean baseline": [],
    "StudyHours": ["StudyHours"],
    "StudyHours + Attendance": ["StudyHours", "Attendance"],
    "StudyHours + Attendance + Age": ["StudyHours", "Attendance", "Age"],
}
def fit(cols):
    if not cols:                                       # no features: predict the training mean
        return DummyRegressor(strategy="mean").fit(train[["StudyHours"]], train["ExamScore"])
    return LinearRegression().fit(train[cols], train["ExamScore"])

val_mae = {}
for name, cols in candidates.items():
    p = fit(cols).predict(validation[cols or ["StudyHours"]])
    val_mae[name] = mean_absolute_error(validation["ExamScore"], p)
    print(f"{name:<30} validation MAE {val_mae[name]:.2f}")
chosen = min(val_mae, key=val_mae.get)
print("Chosen on validation:", chosen)

Mean baseline                  validation MAE 10.59
StudyHours                     validation MAE 9.78
StudyHours + Attendance        validation MAE 8.88
StudyHours + Attendance + Age  validation MAE 8.83
Chosen on validation: StudyHours + Attendance + Age


In [ ]:
# Step 3: evaluate the chosen model and the baseline ONCE on the same test students
def evaluate(name, actual, predicted):
    mse = mean_squared_error(actual, predicted)
    print(f"{name:<30} test MAE {mean_absolute_error(actual, predicted):5.2f} | RMSE {mse ** 0.5:5.2f}"
          f" | R2 {r2_score(actual, predicted):6.3f}")

cols = candidates[chosen]
model = fit(cols)
baseline_pred = fit([]).predict(test[["StudyHours"]])
model_pred = model.predict(test[cols])
evaluate("Mean baseline", test["ExamScore"], baseline_pred)
evaluate(chosen, test["ExamScore"], model_pred)
gain = 1 - mean_absolute_error(test["ExamScore"], model_pred) / mean_absolute_error(test["ExamScore"], baseline_pred)
print(f"MAE reduced by {gain:.0%} on {len(test)} test students")
pd.DataFrame({"ActualScore": test["ExamScore"], "PredictedScore": model_pred.round(1)}).to_csv("test_predictions.csv", index=False)

Mean baseline                  test MAE 10.87 | RMSE 13.92 | R2 -0.002
StudyHours + Attendance + Age  test MAE  8.62 | RMSE  9.92 | R2  0.491
MAE reduced by 21% on 23 test students


In [ ]:
# Step 4: the chosen model's coefficients
print(f"Intercept: {model.intercept_:.2f}")
print(pd.DataFrame({"Feature": cols, "Coefficient": model.coef_.round(2)}).to_string(index=False))

Intercept: -0.70
   Feature  Coefficient
StudyHours         5.49
Attendance         0.39
       Age         0.64


## 6. Mini project: student exam score prediction

In [ ]:
# Mini project. 1-3: load, prepare features and target, split (70/15/15)
df = pd.read_csv("student_data_150.csv")
features = ["StudyHours", "Attendance", "Age"]
print("Rows:", len(df), "| missing values in features and target:", int(df[features + ["ExamScore"]].isna().sum().sum()))
train_val, test = train_test_split(df, test_size=0.15, random_state=42)
train, validation = train_test_split(train_val, test_size=0.15 / 0.85, random_state=42)
print(f"Training ExamScore: mean {train['ExamScore'].mean():.2f} | median {train['ExamScore'].median():.2f}")

Rows: 150 | missing values in features and target: 0
Training ExamScore: mean 57.08 | median 57.00


In [ ]:
# 4-6: baselines and linear regression, compared with all four metrics on VALIDATION
def metrics(actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {"MAE": mean_absolute_error(actual, predicted), "MSE": mse, "RMSE": mse ** 0.5, "R2": r2_score(actual, predicted)}

models = {
    "Mean baseline": (DummyRegressor(strategy="mean"), features),
    "Median baseline": (DummyRegressor(strategy="median"), features),
    "LR: StudyHours": (LinearRegression(), ["StudyHours"]),
    "LR: + Attendance": (LinearRegression(), ["StudyHours", "Attendance"]),
    "LR: + Attendance + Age": (LinearRegression(), features),
}
val = {}
for name, (m, cols) in models.items():
    m.fit(train[cols], train["ExamScore"])
    val[name] = metrics(validation["ExamScore"], m.predict(validation[cols]))
print(pd.DataFrame(val).T.round(3).to_string())
best = min((n for n in models if n.startswith("LR")), key=lambda n: val[n]["MAE"])
print("Best linear regression on validation:", best)

                           MAE      MSE    RMSE     R2
Mean baseline           10.592  162.300  12.740 -0.012
Median baseline         10.609  162.522  12.748 -0.014
LR: StudyHours           9.778  122.451  11.066  0.236
LR: + Attendance         8.880  110.764  10.524  0.309
LR: + Attendance + Age   8.830  110.243  10.500  0.312
Best linear regression on validation: LR: + Attendance + Age


In [ ]:
# Final check: the chosen model and both baselines, ONCE, on the same test students
rows = []
for name in ["Mean baseline", "Median baseline", best]:
    m, cols = models[name]
    label = "Linear regression" if name == best else name            # a short label for the chart
    rows.append({"Model": label, **metrics(test["ExamScore"], m.predict(test[cols]))})
results = pd.DataFrame(rows).round(3)
print(results.to_string(index=False))
results.to_csv("project_results.csv", index=False)

            Model    MAE     MSE   RMSE     R2
    Mean baseline 10.866 193.899 13.925 -0.002
  Median baseline 10.870 194.000 13.928 -0.003
Linear regression  8.615  98.470  9.923  0.491


In [ ]:
# 7: explain the results. Coefficients, then the students the model gets most wrong
model, cols = models[best]
print(f"Intercept {model.intercept_:.2f} |", " | ".join(f"{c} {b:+.2f}" for c, b in zip(cols, model.coef_)))
errors = test.assign(Predicted=model.predict(test[cols]).round(1))
errors["Error"] = (errors["ExamScore"] - errors["Predicted"]).round(1)
print(errors.reindex(errors["Error"].abs().sort_values(ascending=False).index)
      [["Student", "StudyHours", "Attendance", "ExamScore", "Predicted", "Error"]].head(3).to_string(index=False))

Intercept -0.70 | StudyHours +5.49 | Attendance +0.39 | Age +0.64
Student  StudyHours  Attendance  ExamScore  Predicted  Error
   G024         1.5        78.0       62.0       44.8   17.2
   G097         3.5        77.0       73.0       56.1   16.9
   G134         3.0        82.0       72.0       55.2   16.8
